# Homepage Demo
This notebook was created to support the basic example that is showcased on the VeRoViz homepage (https://veroviz.org). 

The goal is to demonstrate a common workflow (create nodes, calculate distances, solve problem, visualize solutions) via a simple example.

---
## Import VeRoViz

We first need to import the VeRoViz Python package:

In [1]:
import veroviz as vrv
import os

vrv.checkVersion()

'Your current installed version of veroviz is 0.4.6, the latest version available is 0.4.5. To update to the latest version, type `pip install --upgrade veroviz` at a command-line prompt.'

## Select a Data Provider

Now, we'll specify a data provider, and any additional arguments required by the chosen data provider.

On the VeRoViz homepage, we used 'OSRM'.  Here, we'll use 'ORS'. 

- See https://veroviz.org/docs/dataproviders.html for other `dataProvider` options.
- See https://veroviz.org/documentation.html for information about setting environment variables (e.g., `os.environ['ORSKEY']`).

In [2]:
# DATA_PROVIDER = 'OSRM-online'
# DATA_PROVIDER_ARGS = {}

DATA_PROVIDER = 'ORS-online'
DATA_PROVIDER_ARGS = {'APIkey': os.environ['ORSKEY'], 'databaseName': None}

## Create a Bounding Region

We're going to automatically generate some random nodes on the map.  To do this, we'll need to define a bounding region, within which the nodes will be created.

The [Sketch](https://veroviz.org/sketch.html) tool makes it easy to create a bounding region.  The [lat, lon] points below were copied from Sketch.

In [3]:
myBoundary = [[42.914949262700084, -78.90020370483398], 
              [42.871938424448466, -78.89556884765626], 
              [42.875083507773496, -78.82158279418947], 
              [42.91293772859624, -78.81729125976564]]

In [4]:
# Create a map that shows just the boundary:
vrv.createLeaflet(boundingRegion = myBoundary)

## Generate Nodes within the Boundary

We'll generate two types of nodes:
1. A single depot (shown with a red "home" icon), and
2. Four customer nodes (with blue "star" icons).

These nodes will be uniformly distributed over the boundary region.

- See https://veroviz.org/docs/veroviz.generateNodes.html for more information on the `generateNodes()` options.

In [5]:
# Single depot node:
nodesDF = vrv.generateNodes(
    numNodes         = 1,
    startNode        = 0,
    nodeType         = 'depot',
    leafletColor     = 'red',
    leafletIconType  = 'home',
    nodeDistrib      = 'uniformBB', 
    nodeDistribArgs  = { 'boundingRegion': myBoundary },
    dataProvider     = DATA_PROVIDER,
    dataProviderArgs = DATA_PROVIDER_ARGS,
    snapToRoad       = True)

In [6]:
# 4 Customer nodes:
nodesDF = vrv.generateNodes(
    initNodes       = nodesDF,
    numNodes        = 4,
    startNode       = 1,
    nodeType        = 'customer',
    leafletColor    = 'blue',
    leafletIconType = 'star',
    nodeDistrib     = 'uniformBB', 
    nodeDistribArgs = { 'boundingRegion': myBoundary },
    dataProvider     = DATA_PROVIDER,
    dataProviderArgs = DATA_PROVIDER_ARGS,
    snapToRoad      = True)

We can now inspect the "nodes" dataframe that was generated.

- Details on the columns in the "nodes" dataframe (as well as information on the other pandas dataframe structures used by VeRoViz) may be found at https://veroviz.org/docs/dataframes.html.

In [7]:
# Display the nodesDF dataframe:
nodesDF

,id,lat,lon,altMeters,nodeName,nodeType,popupText,leafletIconPrefix,leafletIconType,leafletColor,leafletIconText,cesiumIconType,cesiumColor,cesiumIconText,elevMeters
0,0,42.905181,-78.873674,0,None,depot,0,glyphicon,home,red,0,pin,blue,0,None
1,1,42.878841,-78.876142,0,None,customer,1,glyphicon,star,blue,1,pin,blue,1,None
2,2,42.888178,-78.892265,0,None,customer,2,glyphicon,star,blue,2,pin,blue,2,None
3,3,42.883827,-78.879129,0,None,customer,3,glyphicon,star,blue,3,pin,blue,3,None
4,4,42.878182,-78.873413,0,None,customer,4,glyphicon,star,blue,4,pin,blue,4,None


The `createLeaflet()` function can be used to display the boundary region and our nodes.

- See https://veroviz.org/docs/veroviz.createLeaflet.html for more options.

In [8]:
# Create a map that shows the boundary and the nodes:
vrv.createLeaflet(nodes          = nodesDF, 
                  boundingRegion = myBoundary,
                  mapBackground  = 'Arcgis Roadmap')

## Create Travel Matrices

- See https://veroviz.org/docs/veroviz.getTimeDist2D.html for more options.
- There is also a 3D function (https://veroviz.org/docs/veroviz.getTimeDist3D.html), which is suitable for flying vehicles like drones.


In [9]:
[time, dist] = vrv.getTimeDist2D(nodes            = nodesDF, 
                                 matrixType       = 'all2all',
                                 routeType        = 'fastest', 
                                 dataProvider     = DATA_PROVIDER,
                                 dataProviderArgs = DATA_PROVIDER_ARGS)

In [10]:
print("'time' Dictionary (units in seconds): ")
print(time)

print("\nTime from depot (node 0) to customer 3 (node 3): ", time[0,3], "seconds")

print("\n'dist' Dictionary (units in meters):")
print(dist)

print("\nDistance from depot to customer 3: ", dist[0,3], "meters")

'time' Dictionary (units in seconds): 
{(0, 0): 0.0, (0, 1): 593.43, (0, 2): 504.05, (0, 3): 303.58, (0, 4): 403.12, (1, 0): 488.22, (1, 1): 0.0, (1, 2): 870.68, (1, 3): 218.79, (1, 4): 185.56, (2, 0): 577.65, (2, 1): 948.02, (2, 2): 0.0, (2, 3): 747.67, (2, 4): 852.63, (3, 0): 341.47, (3, 1): 314.74, (3, 2): 651.91, (3, 3): 0.0, (3, 4): 150.52, (4, 0): 452.19, (4, 1): 452.5, (4, 2): 696.8, (4, 3): 162.64, (4, 4): 0.0}

Time from depot (node 0) to customer 3 (node 3):  303.58 seconds

'dist' Dictionary (units in meters):
{(0, 0): 0.0, (0, 1): 7458.74, (0, 2): 4028.44, (0, 3): 2799.07, (0, 4): 3905.21, (1, 0): 4488.52, (1, 1): 0.0, (1, 2): 6610.98, (1, 3): 1820.57, (1, 4): 1417.39, (2, 0): 4177.62, (2, 1): 10571.57, (2, 2): 0.0, (2, 3): 4660.01, (2, 4): 6873.84, (3, 0): 3397.99, (3, 1): 4820.75, (3, 2): 4790.47, (3, 3): 0.0, (3, 4): 1218.24, (4, 0): 4551.18, (4, 1): 5988.14, (4, 2): 5538.79, (4, 3): 1328.47, (4, 4): 0.0}

Distance from depot to customer 3:  2799.07 meters


## Generate a dummy solution

This is where the user would typically apply their own algorithm to solve a problem.

For demonstration purposes, let's create a solution "by hand".  The solution should have the following features:
1. There should be two (2) vehicles:
   - One truck, and 
   - One UAV/drone
2. Both vehicles should start/end at the depot node (node 0).
   - The truck should go from the depot to 1 to 2 and back to the depot.
   - The UAV should go from the depot to 3 with a package; from 3 back to the depot empty; from the depot to 4 with a package; and return to the depot empty.
3. Each vehicle should spend 30 seconds at each node.
4. Each vehicle should drop a package at each customer.
5. The truck should follow the road network.
6. The UAV will have a "square" flight path, meaning that it will takeoff and land vertically, and fly horizontally at a constant altitude.  

The `dummySolver()` function will return an "assignments" dataframe.  
- See https://veroviz.org/docs/assignments.html for details on the "assignments" dataframe structure.

In [15]:
def dummySolver(nodesDF, dist, time):
    import pandas as pd
    
    # Assume truck travels depot -> 1 -> 2 -> depot
    route = {'truck': [0, 1, 2, 0], 
             'drone': [0, 3, 0, 4, 0]}
  
    configs = {'truck': {
                    'vehicleModels': ['veroviz/models/ub_truck.glb'],
                    'leafletColor': 'blue',
                    'cesiumColor': 'Cesium.Color.BLUE',
                    'packageModel': 'veroviz/models/box_blue.gltf',
                    'modelScale': 100,
                    'minPxSize': 45 }, 
               'drone': {'vehicleModels': ['veroviz/models/drone.glb', 'veroviz/models/drone_package.glb'],
                    'leafletColor': 'orange',
                    'cesiumColor': 'Cesium.Color.ORANGE',
                    'packageModel': 'veroviz/models/box_yellow.gltf',
                    'modelScale': 100,
                    'minPxSize': 45 }
              }
        
    serviceTime = 30 # seconds    
    
    # Initialize an empty "assignments" dataframe.  
    assignmentsDF = vrv.initDataframe('assignments')

    for vehicle in route:
        startTime = 0
        for i in list(range(0, len(route[vehicle])-1)):
            startNode = route[vehicle][i]
            endNode   = route[vehicle][i+1]

            startLat  = nodesDF[nodesDF['id'] == startNode]['lat'].values[0]
            startLon  = nodesDF[nodesDF['id'] == startNode]['lon'].values[0]
            endLat    = nodesDF[nodesDF['id'] == endNode]['lat'].values[0]
            endLon    = nodesDF[nodesDF['id'] == endNode]['lon'].values[0]

            if ((vehicle == 'drone') and (startNode == 0)):
                # Use the 3D model of a drone carrying a package
                myModel = configs[vehicle]['vehicleModels'][1]
            else:
                # Use the 3D model of either a delivery truck or an empty drone
                myModel = configs[vehicle]['vehicleModels'][0]

            if (vehicle == 'truck'):
                # Get turn-by-turn navigation for the truck, as it travels
                # from the startNode to the endNode:
                shapepointsDF = vrv.getShapepoints2D(
                    # odID           = odID,
                    objectID         = vehicle, 
                    modelFile        = myModel,
                    modelScale       = configs[vehicle]['modelScale'], 
                    modelMinPxSize   = configs[vehicle]['minPxSize'], 
                    startTimeSec     = startTime,
                    startLoc         = [startLat, startLon],
                    endLoc           = [endLat, endLon],
                    # expDurationSec = time[startNode, endNode], 
                    routeType        = 'fastest',
                    leafletColor     = configs[vehicle]['leafletColor'], 
                    # leafletWeight  = 3, 
                    # leafletStyle   = myArcStyle, 
                    # leafletOpacity = 0.8,
                    cesiumColor      = configs[vehicle]['cesiumColor'], 
                    # cesiumWeight   = 3, 
                    # cesiumStyle    = myArcStyle, 
                    # cesiumOpacity  = 0.8,
                    dataProvider     = DATA_PROVIDER,
                    dataProviderArgs = DATA_PROVIDER_ARGS) 
            else:
                # Get a 3D flight profile for the drone:
                shapepointsDF = vrv.getShapepoints3D(
                    # odID             = odID,
                    objectID           = vehicle, 
                    modelFile          = myModel,
                    modelScale         = configs[vehicle]['modelScale'], 
                    modelMinPxSize     = configs[vehicle]['minPxSize'], 
                    startTimeSec       = startTime,
                    startLoc           = [startLat, startLon],
                    endLoc             = [endLat, endLon],
                    takeoffSpeedMPS    = 5,                # FIXME
                    cruiseSpeedMPS     = 20,               # FIXME
                    landSpeedMPS       = 3,                # FIXME
                    cruiseAltMetersAGL = 100,              # FIXME
                    routeType          = 'square',
                    cesiumColor        = configs[vehicle]['cesiumColor']) 
                    # cesiumWeight     = 3, 
                    # cesiumStyle      = myArcStyle, 
                    # cesiumOpacity    = 0.8)
                
            # Update the assignments dataframe:
            assignmentsDF = pd.concat([assignmentsDF, shapepointsDF], ignore_index=True, sort=False)

            # Update the time
            startTime = max(shapepointsDF['endTimeSec'])

            # Add loitering for service
            assignmentsDF = vrv.addStaticAssignment(
                initAssignments      = assignmentsDF, 
                # odID                 = odID, 
                objectID             = vehicle, 
                modelFile            = myModel, 
                modelScale           = configs[vehicle]['modelScale'], 
                modelMinPxSize       = configs[vehicle]['minPxSize'], 
                loc                  = [endLat, endLon],
                startTimeSec    = startTime,
                endTimeSec = startTime + serviceTime)

            # odID += 1

            # Update the time again
            startTime = startTime + serviceTime

            # Add a package at all non-depot nodes:
            if (endNode != 0):
                assignmentsDF = vrv.addStaticAssignment(
                    initAssignments      = assignmentsDF, 
                    # odID                 = 0, 
                    objectID             = 'package %d' % endNode,
                    modelFile            = configs[vehicle]['packageModel'], 
                    modelScale           = 20, 
                    modelMinPxSize       = 30, 
                    loc                  = [endLat, endLon],
                    startTimeSec    = startTime,
                    endTimeSec = -1)
        
    return assignmentsDF

### Call our function to get a solution:

In [16]:
assignmentsDF = dummySolver(nodesDF, dist, time)
assignmentsDF

,odID,objectID,modelFile,modelScale,modelMinPxSize,startTimeSec,startLat,startLon,startAltMeters,endTimeSec,...,ganttColor,popupText,startElevMeters,endElevMeters,wayname,waycategory,surface,waytype,steepness,tollway
0,1,truck,/veroviz/models/ub_truck.glb,100,45,0.000000,42.905181,-78.873674,0,1.334608,...,darkgray,None,204.0,204.0,-,No category,Unknown,None,0,None
1,1,truck,/veroviz/models/ub_truck.glb,100,45,1.334608,42.905183,-78.873742,0,2.363178,...,darkgray,None,204.0,204.0,-,No category,Unknown,None,0,None
2,1,truck,/veroviz/models/ub_truck.glb,100,45,2.363178,42.905174,-78.873793,0,3.311804,...,darkgray,None,204.0,204.0,-,No category,Unknown,None,0,None
3,1,truck,/veroviz/models/ub_truck.glb,100,45,3.311804,42.905148,-78.873826,0,5.949209,...,darkgray,None,204.0,204.0,-,No category,Unknown,None,0,None
4,1,truck,/veroviz/models/ub_truck.glb,100,45,5.949209,42.905051,-78.873852,0,15.500000,...,darkgray,None,204.0,204.0,-,No category,Unknown,None,0,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
425,11,package 4,/veroviz/models/box_yellow.gltf,20,30,641.341794,42.878182,-78.873413,0,-1.000000,...,darkgray,None,None,None,None,None,None,None,None,None
426,1,drone,/veroviz/models/drone.glb,100,45,641.341794,42.878182,-78.873413,0,661.341794,...,darkgray,None,None,None,None,None,None,None,None,None
427,1,drone,/veroviz/models/drone.glb,100,45,661.341794,42.878182,-78.873413,100,811.312373,...,darkgray,None,None,None,None,None,None,None,None,None
428,1,drone,/veroviz/models/drone.glb,100,45,811.312373,42.905181,-78.873674,100,844.645706,...,darkgray,None,None,None,None,None,None,None,None,None


## Visualize the dummy solution

We'll create a static Leaflet map and a dynamic Cesium "movie".

### Static Leaflet map

In [17]:
# Create a Leaflet map showing the nodes and the routes.
vrv.createLeaflet(nodes=nodesDF, arcs=assignmentsDF)

### Dynamic Cesium movie

In [18]:
# Create a Cesium movie showing the nodes, routes, and package deliveries.
vrv.createCesium(
    assignments = assignmentsDF, 
    nodes       = nodesDF, 
    startTime   = '10:00:00', 
    cesiumDir   = os.environ['CESIUMDIR'],
    problemDir  = 'homepage_demo')

Message: File selector was written to /home/cmurray3/cesium/homepage_demo/;homepage_demo.vrv ...
Message: Configs were written to /home/cmurray3/cesium/homepage_demo/config.js ...
Message: Nodes were written to /home/cmurray3/cesium/homepage_demo/displayNodes.js ...
Message: Assignments (.js) were written to /home/cmurray3/cesium/homepage_demo/displayPaths.js ...
Message: Assignments (.czml) were written to /home/cmurray3/cesium/homepage_demo/routes.czml ...
